<a target="_blank" href="https://colab.research.google.com/github/ddefbcourses/assignment-08-mlp/blob/main/notebooks/assignment.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta versão da atividade utilizaremos o dataset CIFAR-10.

Características do dataset:

- 60.000 imagens RGB
- 10 classes
- imagens 32×32
- 3 canais de cor

Importante:

O carregamento do dataset pode ser realizado utilizando:

```python
from tensorflow.keras.datasets import cifar10

(X_train, y_train), (X_test, y_test) = cifar10.load_data()
```

Após o carregamento:

```python
print(X_train.shape)
```

Saída esperada:

```python
(50000, 32, 32, 3)
```

Onde:

- 50000 - número de imagens;
- 32 × 32 - dimensão espacial;
- 3 - canais RGB.

Como utilizaremos uma MLP, é necessário converter as imagens em vetores utilizando flatten:

```python
X_train = X_train.reshape(X_train.shape[0], -1)
X_test = X_test.reshape(X_test.shape[0], -1)
```

Após o flatten:

```python
print(X_train.shape)
```

Saída esperada:

```python
(50000, 3072)
```

Isso ocorre porque:

```python
32 × 32 × 3 = 3072
```

# Objetivos

Nesta atividade você irá:

- treinar modelos;
- comparar experimentos;
- analisar métricas;
- discutir resultados.


Nesta atividade utilizaremos MLflow para:

- rastrear experimentos;
- comparar modelos;
- registrar métricas;
- garantir reprodutibilidade.

In [29]:
import warnings

warnings.filterwarnings("ignore")

In [30]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mlflow

In [31]:
mlflow.set_experiment(
    "assignment"
)

<Experiment: artifact_location='/content/mlruns/1', creation_time=1779470638023, experiment_id='1', last_update_time=1779470638023, lifecycle_stage='active', name='assignment', tags={}, trace_location=None, workspace='default'>

# Questão 1

Implemente uma função `load_data(seed)` que:

- carregue o dataset CIFAR-10 utilizando `tensorflow.keras.datasets.cifar10.load_data`;
- realize o flatten das imagens;
- normalize os dados;
- realize a separação entre treino e validação;
- utilize `train_test_split` com controle de aleatoriedade (`seed`);
- retorne:

```python
X_train, X_val, y_train, y_val
```

já normalizados e preparados para treinamento.

Além disso, responda:

1. Qual o formato original das imagens?
2. Quantas features cada imagem possui após o flatten?
3. Por que o flatten é necessário para uma MLP?
4. Qual a importância da normalização para o treinamento?

**Solução**:

In [ ]:
from tensorflow.keras.datasets import cifar10
from sklearn.model_selection import train_test_split

def load_data(seed):
    (X_train_full, y_train_full), (X_test, y_test) = cifar10.load_data()

    y_train_full = y_train_full.ravel()

    X_train_full = X_train_full.reshape(X_train_full.shape[0], -1)

    X_train_full = X_train_full.astype("float32") / 255.0

    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full,
        y_train_full,
        test_size=0.2,
        random_state=seed,
        stratify=y_train_full
    )

    return X_train, X_val, y_train, y_val

Respostas:

1. O formato original das imagens do CIFAR-10 é 32x32x3
2. Depois do flatten, cada imagem passa a ter 3072 features
3. O flatten é necessário porque uma MLP trabalha com vetores de entrada, e não diretamente com imagens em formato tridimensional. Então, em vez de passar a imagem como uma matriz 32x32 com 3 canais, transformamos tudo em um único vetor com 3072 valores.
4. normalização também é importante porque os pixels originalmente variam de 0 a 255. Ao dividir por 255, os valores ficam entre 0 e 1, o que ajuda o modelo a treinar de forma mais estável.

# Questão 2

Implemente a função:

```python
train_mlp(
    X_train,
    y_train,
    activation,
    hidden_layers,
    learning_rate,
    seed
)
```

## Requisitos

Sua implementação deve:

- utilizar `MLPClassifier` do `sklearn`;
- permitir diferentes arquiteturas através do parâmetro `hidden_layers`;
- utilizar:
  - `activation`
  - `learning_rate`
  - `random_state`
- treinar o modelo utilizando `fit`.

A função deve retornar o modelo treinado.

Além disso, responda:

1. Quantos parâmetros existem na primeira camada?
2. Qual a função da ativação ReLU?
3. Por que MLPs possuem muitos parâmetros ao trabalhar com imagens?

**Solução**:

In [32]:
from sklearn.neural_network import MLPClassifier

def train_mlp(
    X_train,
    y_train,
    activation,
    hidden_layers,
    learning_rate,
    seed
):
    model = MLPClassifier(
        hidden_layer_sizes=hidden_layers,
        activation=activation,
        learning_rate_init=learning_rate,
        random_state=seed,
        max_iter=20
    )

    model.fit(X_train, y_train)

    return model

Respostas:

1. Depende do número de neurônios: 3072 * número de neurônios.
2. A função de ativação ReLU serve para introduzir não linearidade no modelo. Ela transforma valores negativos em zero e mantém os valores positivos. Isso ajuda a rede a aprender padrões mais complexos, porque sem uma função de ativação não linear, a rede acabaria se comportando de forma parecida com um modelo linear, mesmo tendo várias camadas.
3. As MLPs acabam tendo muitos parâmetros quando trabalham com imagens porque cada pixel vira uma feature separada.

# Questão 3

Implemente a função:

```python
evaluate(model, X_test, y_test)
```

Ela deve:

- realizar predições;
- calcular:
  - accuracy;
  - precision;
  - recall;
  - f1-score.

Utilize `sklearn.metrics`.

Além disso:

- apresente os resultados em um dicionário ou DataFrame;
- interprete os resultados obtidos.

Responda:

1. O que a accuracy representa?
2. Qual a diferença entre precision e recall?
3. Em quais situações o f1-score é importante?

**Solução**:

In [33]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average="weighted")
    recall = recall_score(y_test, y_pred, average="weighted")
    f1 = f1_score(y_test, y_pred, average="weighted")

    results = {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1
    }

    return results

1. Acurácia representa a proporção de arcertos comparado ao total no modelo. Nesse caso, acima de 40% representa que o modelo é melhor que aleatório (10%), apesar de não ser ótimo.
2. A precision mede quanto o modelo erra quando ele diz algo, e o recall mede quando do total de uma classe ele conseguiu pegar.
3. Quando é necessário um balanceamento entre o recall e o precision.

# Questão 4

Implemente o rastreamento experimental utilizando MLflow.

## Devem ser registrados:

### Parâmetros

- activation
- hidden_layers
- learning_rate
- max_iter
- batch_size

### Métricas

- accuracy
- precision
- recall
- f1_score
- training_time

Utilize:

```python
mlflow.log_param()
mlflow.log_metric()
```

Ao final:

- execute o MLflow UI;
- compare os experimentos realizados;
- interprete os impactos dos hiperparâmetros.

Responda:

1. Qual experimento apresentou melhor desempenho?
2. Qual configuração apresentou maior estabilidade?
3. Qual o benefício do rastreamento experimental?

**Solução**:

In [34]:
import time
import mlflow
import pandas as pd

from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def run_experiment(
    X_train,
    y_train,
    X_val,
    y_val,
    activation,
    hidden_layers,
    learning_rate,
    max_iter,
    batch_size,
    seed
):
    with mlflow.start_run():
        start_time = time.time()

        model = MLPClassifier(
            hidden_layer_sizes=hidden_layers,
            activation=activation,
            learning_rate_init=learning_rate,
            max_iter=max_iter,
            batch_size=batch_size,
            random_state=seed
        )

        model.fit(X_train, y_train)

        training_time = time.time() - start_time

        y_pred = model.predict(X_val)

        accuracy = accuracy_score(y_val, y_pred)
        precision = precision_score(y_val, y_pred, average="weighted")
        recall = recall_score(y_val, y_pred, average="weighted")
        f1 = f1_score(y_val, y_pred, average="weighted")

        mlflow.log_param("activation", activation)
        mlflow.log_param("hidden_layers", hidden_layers)
        mlflow.log_param("learning_rate", learning_rate)
        mlflow.log_param("max_iter", max_iter)
        mlflow.log_param("batch_size", batch_size)

        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)
        mlflow.log_metric("training_time", training_time)

        results = {
            "activation": activation,
            "hidden_layers": hidden_layers,
            "learning_rate": learning_rate,
            "max_iter": max_iter,
            "batch_size": batch_size,
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1_score": f1,
            "training_time": training_time
        }

        return model, results

In [35]:
experiments = [
    {
        "activation": "relu",
        "hidden_layers": (128,),
        "learning_rate": 0.001,
        "max_iter": 20,
        "batch_size": 128
    },
    {
        "activation": "relu",
        "hidden_layers": (256,),
        "learning_rate": 0.001,
        "max_iter": 20,
        "batch_size": 128
    },
    {
        "activation": "tanh",
        "hidden_layers": (128,),
        "learning_rate": 0.001,
        "max_iter": 20,
        "batch_size": 128
    },
    {
        "activation": "relu",
        "hidden_layers": (128, 64),
        "learning_rate": 0.001,
        "max_iter": 20,
        "batch_size": 128
    },
    {
        "activation": "relu",
        "hidden_layers": (128,),
        "learning_rate": 0.01,
        "max_iter": 20,
        "batch_size": 128
    }
]

In [ ]:
all_results = []

for exp in experiments:
    model, results = run_experiment(
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
        activation=exp["activation"],
        hidden_layers=exp["hidden_layers"],
        learning_rate=exp["learning_rate"],
        max_iter=exp["max_iter"],
        batch_size=exp["batch_size"],
        seed=42
    )

    all_results.append(results)

df_results = pd.DataFrame(all_results)
df_results

Respostas:
1. O experimento que apresentou o melhor desempenho geral foi o experimento: activation = "relu"
hidden_layers = (128, 64)
learning_rate = 0.001
max_iter = 20
batch_size = 128

2. Os que tem learning rate: 0.001

3. O benefício do rastreamento experimental com MLflow é que ele permite comparar os experimentos de forma organizada.

# Questão 5

Compare as funções:

- logistic
- tanh
- relu

## Requisitos

Utilize:

- mesma arquitetura;
- mesmo learning rate;
- mesma seed.

Para cada experimento:

- treine o modelo;
- avalie o modelo;
- registre no MLflow.

Depois compare:

- accuracy;
- convergência;
- estabilidade.

Responda:

1. Qual ativação apresentou melhor convergência?
2. Qual ativação apresentou maior estabilidade?
3. Houve diferenças significativas no treinamento?
4. Por que a ReLU é amplamente utilizada em Deep Learning?

**Solução**:

In [ ]:
import time
import mlflow
import pandas as pd

from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def run_activation_experiment(
    X_train,
    y_train,
    X_val,
    y_val,
    activation,
    hidden_layers=(128,),
    learning_rate=0.001,
    max_iter=20,
    batch_size=128,
    seed=42
):
    with mlflow.start_run(run_name=f"activation_{activation}"):
        start_time = time.time()

        model = MLPClassifier(
            hidden_layer_sizes=hidden_layers,
            activation=activation,
            learning_rate_init=learning_rate,
            max_iter=max_iter,
            batch_size=batch_size,
            random_state=seed
        )

        model.fit(X_train, y_train)

        training_time = time.time() - start_time

        y_pred = model.predict(X_val)

        accuracy = accuracy_score(y_val, y_pred)
        precision = precision_score(y_val, y_pred, average="weighted")
        recall = recall_score(y_val, y_pred, average="weighted")
        f1 = f1_score(y_val, y_pred, average="weighted")

        mlflow.log_param("activation", activation)
        mlflow.log_param("hidden_layers", hidden_layers)
        mlflow.log_param("learning_rate", learning_rate)
        mlflow.log_param("max_iter", max_iter)
        mlflow.log_param("batch_size", batch_size)
        mlflow.log_param("seed", seed)

        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)
        mlflow.log_metric("training_time", training_time)

        results = {
            "activation": activation,
            "hidden_layers": hidden_layers,
            "learning_rate": learning_rate,
            "max_iter": max_iter,
            "batch_size": batch_size,
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1_score": f1,
            "training_time": training_time
        }

        return model, results

In [ ]:
mlflow.set_experiment("MLP_CIFAR10_Activation_Comparison")

activations = ["logistic", "tanh", "relu"]

activation_results = []

for activation in activations:
    model, results = run_activation_experiment(
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
        activation=activation,
        hidden_layers=(128,),
        learning_rate=0.001,
        max_iter=20,
        batch_size=128,
        seed=42
    )

    activation_results.append(results)

df_activation_results = pd.DataFrame(activation_results)
df_activation_results

Respostas:

1. A ativação que apresentou melhor convergência foi a logistic.
2. Também foi a logistic, porque ela teve o melhor equilíbrio entre accuracy, precision, recall e f1-score.
3. Sim, nesse caso a reLU demorou significamente mais pra treinar do que as outras:
4. Mesmo a ReLU não tendo sido a melhor nesse teste específico, ela é amplamente utilizada em Deep Learning porque costuma funcionar muito bem em redes mais profundas. A principal vantagem dela é evitar melhor o problema de saturação em valores positivos, permitindo que os gradientes fluam com mais facilidade durante o treinamento.

# Questão 6

Compare as seguintes arquiteturas:

```python
(32,)
(64,)
(128, 64)
(256, 128)
```

## Requisitos

Para cada arquitetura:

- treine;
- avalie;
- registre no MLflow.

Analise:

- accuracy;
- custo computacional;
- estabilidade;
- overfitting.

Responda:

1. Redes maiores sempre melhoraram os resultados?
2. Qual arquitetura apresentou melhor tradeoff?
3. Houve sinais de overfitting?
4. Qual arquitetura apresentou maior custo computacional?

**Solução**:

In [ ]:
architectures = [
    (32,),
    (64,),
    (128, 64),
    (256, 128)
]

architecture_results = []

mlflow.set_experiment("MLP_CIFAR10_Architecture_Comparison")

for hidden_layers in architectures:
    model, results = run_experiment(
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
        activation="relu",
        hidden_layers=hidden_layers,
        learning_rate=0.001,
        max_iter=20,
        batch_size=128,
        seed=42
    )

    architecture_results.append(results)

df_architecture_results = pd.DataFrame(architecture_results)
df_architecture_results

Respostas:

1. Não, nem sempre
2. arquitetura com melhor tradeoff foi a (128, 64), porque teve a melhor accuracy geral e um tempo de treinamento bem menor que (256, 128).
3. Comm esses dados, não dá para afirmar overfitting com certeza, porque só temos métricas de validação e tempo. Mas existe um possível sinal: a arquitetura (256, 128) é bem maior, treinou por muito mais tempo e não melhorou a accuracy em relação à (128, 64).
4. A arquitetura com maior custo computacional foi a (256, 128).

# Questão 7

Compare os seguintes learning rates:

```python
0.1
0.01
0.001
```

## Requisitos

Utilize:

- mesma arquitetura;
- mesma ativação;
- mesma seed.

Para cada experimento:

- treine;
- avalie;
- registre no MLflow.

Analise:

- estabilidade;
- convergência;
- accuracy;
- comportamento da loss.

Responda:

1. Qual learning rate apresentou melhor desempenho?
2. Qual apresentou maior instabilidade?
3. O que acontece quando o learning rate é muito alto?
4. O que acontece quando o learning rate é muito baixo?

In [ ]:
learning_rates = [0.1, 0.01, 0.001]

lr_results = []

mlflow.set_experiment("MLP_CIFAR10_Learning_Rate_Comparison")

for lr in learning_rates:
    model, results = run_experiment(
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
        activation="relu",
        hidden_layers=(128, 64),
        learning_rate=lr,
        max_iter=20,
        batch_size=128,
        seed=42
    )

    lr_results.append(results)

df_lr_results = pd.DataFrame(lr_results)
df_lr_results

Resultados:

1. O learning rate com melhor desempenho foi o 0.001, porque teve a maior accuracy, com 0.4797, e também o melhor f1-score, com 0.4665. Além disso, foi o menor tempo de treinamento entre os três testes.
2. O learning rate que apresentou maior instabilidade foi o 0.1. Ele ficou com accuracy de 0.1000, que é praticamente o mesmo que chutar aleatoriamente em um problema com 10 classes.
3. Quando o learning rate é muito alto, o modelo dá passos muito grandes durante o treinamento. Com isso, ele pode “passar direto” por boas soluções.
4. Quando o learning rate é muito baixo, o treinamento tende a ser mais estável, mas pode ficar lento demais.

# Questão 8

Com base nos experimentos realizados, escreva uma discussão contendo:

- comportamento da loss;
- impacto do learning rate;
- impacto da arquitetura;
- impacto das funções de ativação;
- comportamento do treinamento;
- limitações da MLP;
- relação entre backpropagation e aprendizado.

Além disso, responda:

1. Qual configuração apresentou melhor resultado final?
2. Quais foram as principais dificuldades observadas?
3. Por que MLPs possuem limitações para imagens?
4. Como o backpropagation contribui para o aprendizado da rede?

Com base nos experimentos, deu para perceber que a loss foi muito afetada pelo learning rate. Quando o learning rate foi muito alto, como 0.1, o modelo praticamente não aprendeu, ficando com accuracy de 0.10, que é quase chute aleatório. Isso indica que a loss provavelmente ficou instável e não conseguiu diminuir bem.

O learning rate teve um dos maiores impactos no treinamento. O valor 0.001 apresentou o melhor resultado, com accuracy de 0.4797, enquanto 0.01 teve desempenho menor e 0.1 foi muito instável.

A arquitetura também influenciou bastante. Redes muito pequenas, como (32,) e (64,), tiveram desempenho baixo. Já arquiteturas maiores, como (128, 64) e (256, 128), melhoraram os resultados, mas a maior rede teve custo computacional bem mais alto sem melhorar muito a accuracy.

Nas funções de ativação, a logistic teve o melhor resultado entre as três testadas, com accuracy de 0.4750. A ReLU também teve bom desempenho, mas foi mais lenta nesse experimento. A tanh ficou um pouco abaixo das outras.

No geral, o treinamento mostrou que a MLP consegue aprender alguns padrões do CIFAR-10, mas tem limitações. Como as imagens são transformadas em vetores, o modelo perde a noção espacial da imagem, como bordas, regiões próximas e formas. Por isso, MLPs não são tão boas para imagens quanto CNNs.

1. A melhor configuração final foi:


*   activation = "relu"
*   hidden_layers = (128, 64)
*   learning_rate = 0.001
*   max_iter = 20
*   batch_size = 128

Ela teve a melhor accuracy geral, com 0.4797.

2. As principais dificuldades observadas foram o alto tempo de treinamento, a sensibilidade ao learning rate e o limite de desempenho da MLP para imagens. Também ficou claro que aumentar muito a arquitetura nem sempre melhora o resultado.

3. MLPs possuem limitações para imagens porque, ao aplicar o flatten, a estrutura espacial da imagem é perdida. Ou seja, o modelo não entende bem relações locais entre pixels, como bordas, formas e texturas, por isso costuma ter desempenho pior que CNNs em problemas de imagem.

4. O backpropagation contribui para o aprendizado porque calcula o erro da rede e ajusta os pesos das camadas para reduzir esse erro ao longo do treinamento. Assim, a rede vai aprendendo aos poucos quais padrões ajudam mais na classificação.

In [ ]:
# TODO: implemente